# CSCI E-89 Deep Learning — Assignment 03

**Name:** Jazmyn Stokes
**Problem 1 (25%):** Use Claude Code to incrementally build a PyTorch image classifier
for Fashion MNIST (the "Building an Image Classifier with PyTorch" section of
`10_neural_nets_with_pytorch.ipynb`, Chapter 10 of *Hands-On Machine Learning with
Scikit-Learn and PyTorch*, Aurelien Geron, 2025), adding a training-accuracy plot.

Each generated script from the incremental Claude Code session is captured under
`scripts/` in this repository. Each section below is one prompt from that session;
its code cell simply runs (`%run`) the corresponding captured script, in order, so
the notebook cells and the repository's generated scripts are the same code — not a
separate copy of it. Cells all run top to bottom in one shared namespace, so later
sections can use variables/functions defined by earlier ones.


## Prompt log

| # | Prompt to Claude Code | What it produced |
|---|---|---|
| 1 | Setup: imports, pick best device, seed 42, matplotlib defaults, print torch version + device | Imports / device / seed / plot defaults (below) |
| 2 | Load Fashion MNIST with TorchVision, convert to [0,1] float tensors, re-seed 42, split 55k/5k train/validation, print split sizes + class names | Dataset loading + split (below) |
| 3 | Wrap the three splits in DataLoaders (batch_size=32, shuffle train only), print one sample's shape/dtype/label | DataLoaders + sample check (below) |
| 4 | Define the `ImageClassifier` MLP (784/300/100/10), move it to device, create the loss, print the model + parameter count | Model + loss (below) |
| 5 | Define `evaluate_tm()` and `train2()` (returns a history dict), no training run yet | Training/eval helper functions (below) |
| 6 | Train for 20 epochs with SGD(lr=0.1) + torchmetrics multiclass accuracy, keep the returned history | Training run (below) |
| 7 | Plot training vs. validation accuracy per epoch (required addition) + a second figure for training loss | Accuracy/loss plots (below) |
| 8 | Predict 3 validation images, softmax + top-4 probabilities (mps->cpu workaround for round()), show the images | Prediction + visualization (below) |
| 9 | Final evaluation on the held-out test set, print test accuracy + parameter count | Test evaluation (below) |


---
## 1. Setup — imports, device selection, seed, plotting defaults

**Prompt:** "imports (numpy, torch, nn, F, torchmetrics, matplotlib), pick the best
device (Mac), seed 42, set some matplotlib defaults, and print the torch version and
device."

- Imports cover tensors/arrays (`numpy`, `torch`), building blocks for the model
  (`torch.nn` as `nn`, `torch.nn.functional` as `F`), metric tracking (`torchmetrics`),
  and plotting (`matplotlib.pyplot`).
- Device selection checks CUDA first (a discrete/cloud GPU), then Apple Silicon's
  Metal backend (`mps`, what a Mac with an M-series chip uses), then falls back to
  `cpu` everywhere else — this makes the notebook portable across machines.
- `torch.manual_seed(42)` (plus `np.random.seed(42)` for any NumPy-side randomness)
  is set once, up front, so later steps such as the train/validation split, weight
  initialization, and DataLoader shuffling are reproducible across reruns.
- Matplotlib rc defaults set a consistent, readable font/label/legend size for every
  figure drawn later in the notebook (e.g. the training-accuracy plot in a later
  section), instead of repeating `plt.rc(...)` calls near each plot.

Running `scripts/01_setup.py` also runs the sanity check that was folded into it (confirms the imports resolved, a tensor can be moved to `device`, and the seed is reproducible) before anything else is built on top of it.

In [ ]:
%run scripts/01_setup.py


---
## 2. Load Fashion MNIST, transform, and split into train / validation

**Prompt:** "load Fashion MNIST from torchvision into a datasets/ folder, convert to
float tensors scaled to [0,1] with transforms.v2. Re-seed to 42, then split the 60k
training images into 55k train / 5k validation. Print the split sizes and the class
names."

- `torchvision.datasets.FashionMNIST` downloads (or reuses, if already cached) the
  dataset into a local `datasets/` folder. `train=True` gives the 60,000-image
  training pool; `train=False` gives the 10,000-image held-out test set, which is
  loaded here but not touched again until final evaluation.
- The `transforms.v2` pipeline (`ToImage()` then `ToDtype(torch.float32, scale=True)`)
  turns each raw PIL/uint8 image into a `[1, 28, 28]` float32 tensor with pixel
  values scaled from `[0, 255]` down to `[0, 1]`. Unscaled 0-255 inputs would make
  gradient descent unstable at the learning rate used later.
- `torch.manual_seed(42)` is called again immediately before the split, so the exact
  same 55,000/5,000 partition is drawn no matter which earlier cells were or weren't
  re-run first (this is a separate, explicit re-seed as the prompt asked for, on top
  of the one already set in the setup cell).
- `torch.utils.data.random_split` divides the 60,000 training images into 55,000 for
  training and 5,000 held out for validation during training.
- The dataset's `.classes` attribute gives the 10 human-readable Fashion MNIST
  category names (e.g. "T-shirt/top", "Sneaker"), which later cells use to label
  predictions.


In [ ]:
%run scripts/02_load_data.py


---
## 3. DataLoaders for train / validation / test

**Prompt:** "DataLoaders for all three splits, batch_size=32, shuffle the training
one only. Then print one sample's shape, dtype and label so I can see what's going
in."

- `DataLoader` wraps each split so training/evaluation can iterate over mini-batches
  instead of single images. `batch_size=32` matches the assignment's template
  pipeline.
- `shuffle=True` only on the training loader: reshuffling each epoch decorrelates
  consecutive mini-batches, which helps SGD converge. Validation and test are always
  evaluated over the full set regardless of order, so shuffling them would have no
  benefit and only adds overhead.
- Printing one raw sample (straight from `train_data`, before batching) confirms the
  transform from Step 2 produced the expected `[1, 28, 28]` float32 tensor scaled to
  `[0, 1]`, and that the label lines up with a valid `class_names` entry, before any
  model sees the data.


In [ ]:
%run scripts/03_dataloaders.py


---
## 4. The model: `ImageClassifier` (MLP) and the loss function

**Prompt:** "an nn.Module called ImageClassifier: Flatten, Linear, ReLU, Linear,
ReLU, Linear, no activation at the end since CrossEntropyLoss wants logits. Build it
with 784/300/100/10, move it to device, make the loss, and print the model plus the
parameter count."

- `ImageClassifier` subclasses `nn.Module` and defines the forward pass as an
  `nn.Sequential` stack: `Flatten` turns each `[1, 28, 28]` image into a 784-length
  vector, then two hidden `Linear + ReLU` blocks (784→300→100), then a final
  `Linear(100, 10)` output layer with **no activation function**.
- No activation on the output layer is deliberate: `nn.CrossEntropyLoss` applies
  `log_softmax` internally and expects raw, unnormalized logits as input. Adding a
  softmax here would apply it twice and slow down learning.
- The model is moved to `device` (selected in Step 1) with `.to(device)` so its
  parameters live on the same device the input batches will be moved to during
  training/evaluation — a device mismatch between model and data raises an error.
- `nn.CrossEntropyLoss()` is the standard multi-class classification loss, combining
  log-softmax and negative log-likelihood in one call.
- Printing the model shows the layer stack; the parameter count (784*300 + 300 +
  300*100 + 100 + 100*10 + 10 = 266,610) is a sanity check against the assignment's
  reference architecture.


In [ ]:
%run scripts/04_model.py


---
## 5. Training and evaluation helper functions

**Prompt:** "just the functions, don't run anything yet. An evaluate_tm() that runs
a loader under no_grad and returns the metric, and a train2() that trains and
returns a history dict with train_losses, train_metrics and valid_metrics. I need
that history returned, not just printed — I'm plotting it later. Print the metrics
each epoch too."

Only the two function definitions are added in this cell — nothing is executed or
trained yet (that's the next step, once an optimizer exists).

- `evaluate_tm(model, data_loader, metric)` switches the model to `eval()` mode
  (disables dropout / uses running batch-norm stats — not used by this MLP, but it's
  the correct habit), resets the `torchmetrics` metric so no stale state leaks in
  from a previous call, then loops over the loader under `torch.no_grad()` (no
  autograd graph needed for evaluation, which saves memory and time) accumulating
  predictions into the metric. It returns the aggregated metric value via
  `metric.compute()`.
- `train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs)`
  runs the standard training loop for `n_epochs`: for each batch, zero the
  gradients, forward pass, compute loss, backward pass, optimizer step, and
  accumulate both the running loss and the training metric. At the end of each
  epoch it computes the epoch's average training loss and training metric, then
  calls `evaluate_tm` on `valid_loader` to get the validation metric, prints all
  three, and appends them to a `history` dict with keys `train_losses`,
  `train_metrics`, and `valid_metrics`. Returning `history` (rather than only
  printing) is what makes the training-accuracy plot in a later step possible.


In [ ]:
%run scripts/05_train_eval_functions.py


---
## 6. Run training — 20 epochs, SGD(lr=0.1)

**Prompt:** "train for 20 epochs with SGD lr=0.1 and a torchmetrics multiclass
accuracy on the same device. Keep the history."

- `torch.optim.SGD(model.parameters(), lr=0.1)` is plain (non-momentum) stochastic
  gradient descent at a learning rate of 0.1, matching the assignment's reference
  pipeline.
- `torchmetrics.Accuracy(task="multiclass", num_classes=10)` computes classification
  accuracy across the 10 Fashion MNIST classes. It's moved to `device` with
  `.to(device)` — a torchmetrics metric must live on the same device as the tensors
  it's updated with, or it raises a device-mismatch error.
- `train2(...)` (defined in Step 5) is called with `n_epochs=20`; its return value is
  kept in `history` — this is the actual, real training run, and this is the one
  long-running cell in the notebook. On a Mac with Apple Silicon (`mps`) or a GPU,
  20 epochs typically takes a few minutes; longer on CPU.
- `history` is what the next step plots, so nothing here should be re-run without
  also expecting the training-accuracy plot to change.


In [ ]:
%run scripts/06_run_training.py


---
## 7. Plot training accuracy (required) and training loss

**Prompt:** "add code that will plot the training accuracy per epoch, with
validation accuracy on the same plot. Put training at the epoch midpoint since it's
an average over the epoch. Labels, grid, legend, y range 0.7-1.0. Print the final
numbers, and add a second figure for the training loss."

This is the one addition Assignment 03 explicitly requires on top of the book's
reference pipeline.

- Training accuracy is an average computed *over the course of* each epoch (it
  accumulates as each mini-batch is seen), so it's plotted at `epoch + 0.5` — the
  epoch's midpoint — to reflect that it's a running average, not a single point-in-
  time measurement.
- Validation accuracy, in contrast, is measured once, in full, *after* the epoch
  finishes, so it's plotted at `epoch + 1.0` — the epoch's end.
- Labels, a grid, and a legend are added so the plot is self-explanatory; the y-axis
  is fixed to `[0.7, 1.0]` so run-to-run comparisons share the same scale and small
  differences near the top of the range remain visible.
- The final training and validation accuracy values are printed numerically under
  the plot, not just shown visually.
- A second figure plots training loss per epoch as a cross-check: loss falling while
  accuracy rises is the expected, healthy pattern; if they diverge, something in the
  training loop deserves a second look.


In [ ]:
%run scripts/07_plot_accuracy.py


---
## 8. Predict on 3 validation images and visualize

**Prompt:** "predict on 3 images from the validation loader, print predicted vs
actual class names. Then softmax the logits and show the probabilities rounded,
plus the top 4 per image. Heads up: round() isn't implemented on mps, so move to
cpu first. Then show the 3 images with their labels."

- The model is switched to `eval()` mode and one batch is pulled from
  `valid_loader`; only the first 3 images/labels of that batch are used.
- Raw logits from the model are converted to predicted class indices with
  `argmax(dim=1)`, then mapped back to human-readable names via `class_names`, and
  printed alongside the true labels for a quick correct/incorrect check.
- `F.softmax(logits, dim=1)` turns the logits into per-class probabilities.
  **`Tensor.round(decimals=...)` is not implemented on Apple Silicon's `mps`
  backend**, so whenever `device == "mps"` the tensor is moved to `cpu` first with
  `.cpu()` before rounding — this keeps the notebook from crashing on a Mac while
  still working unchanged on `cuda`/`cpu`.
- `torch.topk(logits, k=4, dim=1)` finds each image's 4 highest-scoring classes;
  those top-4 logits are re-softmaxed on their own (`F.softmax` over just the 4
  values) so the reported top-4 probabilities sum to 1 for easy reading, rather than
  showing raw slices of the full 10-way distribution.
- Finally, the 3 images are displayed with `imshow` (moved to `cpu` for plotting,
  since matplotlib only works with CPU/NumPy arrays), each titled with its predicted
  and true class name.

Show the 3 images alongside their predicted and true labels, so the numbers above
can be checked visually against the actual Fashion MNIST items.


In [ ]:
%run scripts/08_predict.py


---
## 9. Final evaluation on the test set

**Prompt:** "one final run on the test set, print the test accuracy and the
parameter count."

- `test_loader` (built in Step 3) has not been touched anywhere in training or
  validation — it's used here for the first and only time, which is what makes this
  a fair, held-out estimate of how the model generalizes.
- Reuses `evaluate_tm` (Step 5) with the same `accuracy` metric object (Step 6);
  `evaluate_tm` resets the metric internally before accumulating over the test set,
  so no state leaks in from the training/validation runs.
- The parameter count is recomputed the same way as in Step 4, as a final summary
  next to the accuracy number.


In [ ]:
%run scripts/09_evaluate.py
